<a href="https://colab.research.google.com/github/rayyan4676t7/credit-card-fraud-detection/blob/main/credit_card_fraud_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Fraud Detection using Machine Learning

## Introduction
Fraud detection is a critical problem in financial systems where the goal is to identify fraudulent transactions.
Due to the increasing number of digital transactions, detecting fraud automatically using machine learning has become essential.

This project aims to detect fraudulent transactions using anomaly detection techniques.

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing
import matplotlib.pyplot as plt
import seaborn as sns
! pip install plotly
import plotly.express as px



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Import & Load Data

In [2]:
df = pd.read_csv("creditcard.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

## Dataset Description

The dataset contains credit card transactions.

- Class = 0 → Normal transaction  
- Class = 1 → Fraud transaction  

The dataset is highly imbalanced, meaning fraud cases are very rare compared to normal transactions.

## Basic Info

In [ ]:
df.head(15)

In [ ]:
df.tail(15)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check missing values
df.isnull().sum()


In [ ]:
df = df.dropna()

## boxplot : Amount vs Fraud

In [ ]:
sns.boxplot(x='Class', y='Amount', data=df)


## Feature Distributions

In [ ]:
plt.figure(figsize=(12, 8))
df.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

## Heatmap

In [ ]:
numeric_df = df.select_dtypes(include='number')

corr_matrix = numeric_df.corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    aspect='auto',
    title='✨ Correlation Heatmap of Numeric Features (Credit Card Fraud)'
)

fig.update_layout(
    template='plotly_white',
    height=800
)

fig.show()

## MRP

In [ ]:
import plotly.express as px

fig = px.box(
    df,
    x='Class',
    y='Amount',
    color='Class',
    color_discrete_map={0: 'teal', 1: 'pink'},
    points="all",
    title="Transaction Amount Distribution by Class"
)

fig.update_layout(
    xaxis_title="Class (0=Normal, 1=Fraud)",
    yaxis_title="Transaction Amount",
    template='plotly_white'
)

fig.show()

## barchart

In [ ]:
top_classes = df['Class'].value_counts().reset_index()
top_classes.columns = ['Class', 'Count']

fig = px.bar(
    top_classes,
    x='Count',
    y='Class',
    orientation='h',
    color='Class',
    color_discrete_map={0: 'teal', 1: 'pink'},
    title='Number of Transactions by Class'
)

fig.update_layout(
    height=400,
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title='Number of Transactions',
    yaxis_title='Class',
    template='plotly_white'
)

fig.show()

## scatter

In [ ]:
df_sample = df.sample(n=5000, random_state=42)

fig = px.scatter(
    df_sample,
    x='Time',
    y='Amount',
    color='Class',
    color_discrete_map={0: 'teal', 1: 'pink'},
    hover_data=['V1', 'V2', 'V3', 'V4', 'V5'],
    title='Transaction Amount vs Time by Class (Sample)'
)

fig.update_layout(
    xaxis_title='Time (seconds)',
    yaxis_title='Transaction Amount',
    template='plotly_white',
    height=600
)

fig.show()

## Bubble Scatter

In [ ]:
df_sample = df.sample(n=5000, random_state=42)

df_sample['V1_size'] = df_sample['V1'].abs() + 0.1


fig = px.scatter(
    df_sample,
    x='Time',
    y='Amount',
    size='V1_size',
    color='Class',
    hover_data=['V1', 'V2', 'V3', 'V4', 'V5'],
    title='Transaction Amount vs Time by Class',
    size_max=60,
    color_discrete_map={0: 'teal', 1: 'pink'}
)

fig.update_layout(
    xaxis_title='Time (seconds)',
    yaxis_title='Transaction Amount',
    template='plotly_white',
    height=600
)

fig.show()



## plotly line

In [ ]:
import plotly.express as px

# Sample data
df_sample = df.sample(n=10000, random_state=42)

# Sort by time
df_sample_sorted = df_sample.sort_values('Time')

# Create smoothed column (BEST method)
df_sample_sorted['Amount_Smooth'] = df_sample_sorted.groupby('Class')['Amount'] \
    .transform(lambda x: x.rolling(window=50, min_periods=1).mean())

# Plot
fig = px.line(
    df_sample_sorted,
    x='Time',
    y='Amount_Smooth',
    color='Class',
    line_dash='Class',
    markers=True,
    title='Smoothed Transaction Amount by Time and Class'
)

fig.update_layout(
    xaxis_title='Time (seconds)',
    yaxis_title='Transaction Amount (Smoothed)',
    template='plotly_white',
    height=600
)

fig.show()

## Pie Chart

In [ ]:
top_classes = df['Class'].value_counts().reset_index()
top_classes.columns = ['Class', 'Count']

fig = px.pie(
    top_classes,
    names='Class',
    values='Count',
    title='Distribution of Transactions by Class',
    color='Class',
    color_discrete_map={0: 'teal', 1: 'pink'}
)


pull = [0.1 if i == 0 else 0 for i in range(len(top_classes))]
fig.update_traces(textinfo='percent+label', pull=pull)

fig.update_layout(legend_title_text='Class')

fig.show()

In [ ]:
df = df.dropna()

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Model Used

Logistic Regression was used as the classification model.

It is a supervised learning algorithm that predicts whether a transaction is fraudulent or not.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Normal Logistic Regression:\n")
print(classification_report(y_test, y_pred))

In [ ]:
model_balanced = LogisticRegression(max_iter=2000, class_weight="balanced")
model_balanced.fit(X_train, y_train)

y_pred_bal = model_balanced.predict(X_test)

print("Balanced Logistic Regression:\n")
print(classification_report(y_test, y_pred_bal))

## Results Interpretation

The dataset is highly imbalanced, so accuracy is not a reliable metric.

The normal model achieved good precision but missed some fraud cases.

After applying class balancing, recall improved, meaning more fraud cases were detected.

However, precision decreased, leading to more false positives.

This demonstrates the trade-off between precision and recall.

## Conclusion

This project demonstrates the challenges of fraud detection using machine learning.

Due to class imbalance, accuracy is not a reliable metric.

Different models and techniques result in trade-offs between precision and recall.

Choosing the right model depends on the application requirements, especially in critical domains like fraud detection.